# Ablation A: EWC Lambda (λ)
Tests sensitivity of EWC penalty strength: λ ∈ {100, 1000, 5000}.

**GPU required.**

### Inputs
- `pi-detection-data`
- `pi-detection-utils`

### Outputs
- `results/results_ablation_lambda.json`

## 1. Install & Setup

In [1]:
!pip install -q transformers accelerate scikit-learn matplotlib seaborn

import os, sys, gc
import torch

KAGGLE_INPUT = '/kaggle/input/datasets/dabiraomotoso'
UTILS_DATASET = os.path.join(KAGGLE_INPUT, 'pi-detection-utils')
for _root in (UTILS_DATASET, os.path.join(UTILS_DATASET, 'antidote')):
    if os.path.isfile(os.path.join(_root, 'utils.py')):
        sys.path.append(_root)
        break
else:
    raise FileNotFoundError(f'No utils.py under {UTILS_DATASET}')

from utils import *

os.makedirs(CFG['checkpoint_dir'], exist_ok=True)
os.makedirs(CFG['results_dir'], exist_ok=True)
os.makedirs('/kaggle/working/figures', exist_ok=True)

print(f'Device: {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

Device: cuda
GPU: Tesla T4
VRAM: 15.6 GB


## 2. Load Data

In [2]:
import os
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("HF_TOKEN")
login(secret_value_0)

In [3]:
import pandas as pd
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(CFG['model_name'])
data_dir  = '/kaggle/input/datasets/dabiraomotoso/pi-detection-data'
tasks     = {}

for task_name, fname in [
    ('T1_LLMail',      't1_llmail.parquet'),
    ('T2_HackAPrompt', 't2_hackaprompt.parquet'),
    ('T3_BIPIA',       't3_bipia.parquet'),
]:
    path = os.path.join(data_dir, fname)
    if os.path.exists(path):
        df = pd.read_parquet(path)
        tr, va, te = make_loaders(df, tokenizer)
        tasks[task_name] = {'train': tr, 'val': va, 'test': te, 'df': df}
        print(f'  {task_name}: loaded')
    else:
        print(f'  {task_name}: NOT FOUND')

print(f'Loaded {len(tasks)} tasks.')

config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

  T1_LLMail: loaded
  T2_HackAPrompt: loaded
  T3_BIPIA: loaded
Loaded 3 tasks.


## 3. OOM Patches

In [4]:
import gc, os
import utils as _utils

os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'

# Patch 1: flush GPU after checkpoint save
original_save = _utils.save_checkpoint

def save_checkpoint_and_flush(model, path):
    original_save(model, path)
    gc.collect()
    torch.cuda.empty_cache()
    free = (torch.cuda.get_device_properties(0).total_memory
            - torch.cuda.memory_allocated()) / 1e9
    print(f'  GPU cache cleared. Free VRAM: {free:.2f}GB')

_utils.save_checkpoint = save_checkpoint_and_flush

# Patch 2: gradient checkpointing to halve activation memory
original_load_model = _utils.load_model

def load_model_with_gc():
    model = original_load_model()
    model.gradient_checkpointing_enable()
    print('  Gradient checkpointing enabled.')
    return model

_utils.load_model = load_model_with_gc

# Patch 3: clear GPU between tasks (correct scoping — patches inside run_experiment)
original_run_experiment = _utils.run_experiment
original_tt = _utils.train_task

def run_experiment_with_cleanup(experiment_name, tasks, tokenizer,
                                 use_ewc=False, use_replay=False, joint_training=False):
    def train_task_patched(*args, **kwargs):
        result = original_tt(*args, **kwargs)
        gc.collect()
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
        free = (torch.cuda.get_device_properties(0).total_memory
                - torch.cuda.memory_allocated()) / 1e9
        print(f'  Inter-task GPU cleared. Free VRAM: {free:.2f}GB')
        return result

    _utils.train_task = train_task_patched
    try:
        result = original_run_experiment(
            experiment_name, tasks, tokenizer,
            use_ewc=use_ewc, use_replay=use_replay, joint_training=joint_training
        )
    finally:
        _utils.train_task = original_tt

    return result

_utils.run_experiment = run_experiment_with_cleanup
run_experiment = run_experiment_with_cleanup

print('All patches active.')

All patches active.


## 4. Run Ablation A
Tests sensitivity of EWC penalty strength.
- Too small → insufficient protection against forgetting
- Too large → underfits new tasks (plasticity sacrificed for stability)
- Sweet spot is what we report as the proposed method

In [5]:
ablation_lambda_results = {}

for lam in [100, 1000, 5000]:
    CFG['ewc_lambda'] = lam
    key = f'ewc_lambda_{lam}'
    print(f'\n--- Running λ={lam} ---')
    ablation_lambda_results[key], _model = run_experiment(
        key, tasks, tokenizer,
        use_ewc=True, use_replay=True, joint_training=False,
    )
    del _model
    gc.collect()
    torch.cuda.empty_cache()
    save_results(ablation_lambda_results, f'{CFG["results_dir"]}/results_ablation_lambda.json')

CFG['ewc_lambda'] = 1000  # reset to default
print('\nAblation A complete. Results saved.')


--- Running λ=100 ---

EXPERIMENT: ewc_lambda_100
  EWC: True | Replay: True | Joint: False


`torch_dtype` is deprecated! Use `dtype` instead!


pytorch_model.bin:   0%|          | 0.00/371M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
classifier.weight                       | MISSING    | 
classifier.bias                         | MISSING    | 
pooler.dense.weight                     | MISSING    | 
pooler.dense.bias        

...

DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
classifier.weight                       | MISSING    | 
classifier.bias                         | MISSING    | 
pooler.dense.weight                     | MISSING    | 
pooler.dense.bias        

  Gradient checkpointing enabled.

--- Training on T1_LLMail (step 1/3) ---
  Epoch 1/3 | Train Loss: 0.1480 | Val F1: 0.9875 | Val Loss: 0.0665
  Epoch 2/3 | Train Loss: 0.0472 | Val F1: 0.9895 | Val Loss: 0.0615
  Epoch 3/3 | Train Loss: 0.0301 | Val F1: 0.9918 | Val Loss: 0.0460
  Best val F1: 0.9918
  Inter-task GPU cleared. Free VRAM: 14.14GB
  Computing Fisher matrix for T1_LLMail...
  EWC registered T1_LLMail. Total tasks: 1
  Replay buffer: +5000 from T1_LLMail. Total: 5000
  After T1_LLMail → T1_LLMail test F1: 0.9929
  Checkpoint saved: /kaggle/working/checkpoints/ewc_lambda_1000_after_T1_LLMail.pt
  GPU cache cleared. Free VRAM: 12.63GB

--- Training on T2_HackAPrompt (step 2/3) ---
  Epoch 1/3 | Train Loss: 0.7874 | Val F1: 0.6616 | Val Loss: 0.5779
  Epoch 2/3 | Train Loss: 0.5709 | Val F1: 0.7609 | Val Loss: 0.5506
  Epoch 3/3 | Train Loss: 0.5155 | Val F1: 0.7657 | Val Loss: 0.5293
  Best val F1: 0.7657
  Inter-task GPU cleared. Free VRAM: 12.63GB
  Computing Fisher matr

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
classifier.weight                       | MISSING    | 
classifier.bias                         | MISSING    | 
pooler.dense.weight                     | MISSING    | 
pooler.dense.bias        

  Gradient checkpointing enabled.

--- Training on T1_LLMail (step 1/3) ---
  Epoch 1/3 | Train Loss: 0.1440 | Val F1: 0.9882 | Val Loss: 0.0634
  Epoch 2/3 | Train Loss: 0.0402 | Val F1: 0.9911 | Val Loss: 0.0458
  Epoch 3/3 | Train Loss: 0.0222 | Val F1: 0.9918 | Val Loss: 0.0484
  Best val F1: 0.9918
  Inter-task GPU cleared. Free VRAM: 14.14GB
  Computing Fisher matrix for T1_LLMail...
  EWC registered T1_LLMail. Total tasks: 1
  Replay buffer: +5000 from T1_LLMail. Total: 5000
  After T1_LLMail → T1_LLMail test F1: 0.9939
  Checkpoint saved: /kaggle/working/checkpoints/ewc_lambda_5000_after_T1_LLMail.pt
  GPU cache cleared. Free VRAM: 12.64GB

--- Training on T2_HackAPrompt (step 2/3) ---
  Epoch 1/3 | Train Loss: 0.8053 | Val F1: 0.6729 | Val Loss: 0.5822
  Epoch 2/3 | Train Loss: 0.5921 | Val F1: 0.7347 | Val Loss: 0.5495
  Epoch 3/3 | Train Loss: 0.5303 | Val F1: 0.7555 | Val Loss: 0.5330
  Best val F1: 0.7555
  Inter-task GPU cleared. Free VRAM: 12.64GB
  Computing Fisher matr